In [1]:
!pip install mujoco mediapy
import mujoco
import mediapy as media
import os

# 1. Force the headless EGL backend (or "osmesa" if you don't have a GPU)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# 2. MANUALLY trigger the GL context initialization to override GLFW
from mujoco import viewer
try:
    # This forces MuJoCo to initialize its GL context using the backend we set above
    mujoco.GLContext(640, 480).make_current()
except Exception:
    pass

/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65537) b'The GLFW library is not initialized'
  warnings.warn(message, GLFWError)


In [2]:
import os
import ctypes
import ctypes.util

# 1. Force the process to globally link the OpenGL library before MuJoCo touches it
try:
    gl_lib = ctypes.util.find_library('GL')
    if gl_lib:
        ctypes.CDLL(gl_lib, ctypes.RTLD_GLOBAL)
except Exception:
    pass

# 2. Tell MuJoCo explicitly to use EGL headless rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# 3. Import and create your renderer safely
import mujoco

# --- Your code below ---
# model = mujoco.MjModel.from_xml_string(...)
# renderer = mujoco.Renderer(model, height=480, width=640)
# print("Renderer successfully initialized!")

In [4]:
import mediapy as media

In [5]:
# 2. DEFINICIÓN DEL MODELO EN XML (Un péndulo doble)
# MuJoCo usa un formato limpio llamado MJCF
# 2. DEFINICIÓN DEL MODELO EN XML (Un péndulo doble corregido)
xml_robot = """
<mujoco>
    <option gravity="0 0 -9.81"/>
    <worldbody>
        <light pos="0 0 3" dir="0 0 -1"/>
        <geom type="plane" size="5 5 0.1" rgba=".9 .9 .9 1"/>

        <!-- Eslabón 1 -->
        <body name="link1" pos="0 0 2">
            <joint name="joint1" type="hinge" axis="0 1 0" pos="0 0 0"/>
            <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" rgba="0 0.7 0 1"/>

            <!-- Eslabón 2 -->
            <body name="link2" pos="0 0 -0.5">
                <joint name="joint2" type="hinge" axis="0 1 0" pos="0 0 0"/>
                <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" rgba="0.7 0 0 1"/>
            </body>
        </body>
    </worldbody>
</mujoco>
"""

# 3. CARGAR EL MODELO Y CREAR EL CONTENEDOR DE DATOS
model = mujoco.MjModel.from_xml_string(xml_robot)
data = mujoco.MjData(model)

# Instanciamos el renderizador de MuJoCo
renderer = mujoco.Renderer(model, height=480, width=640)

# 4. CONDICIONES INICIALES (Ángulos en radianes)
# Sacamos el péndulo del equilibrio para que caiga por gravedad
data.qpos[0] = 1.57  # ~90 grados para el eslabón 1
data.qpos[1] = 1.0   # ~57 grados para el eslabón 2

# 5. BUCLE DE SIMULACIÓN Y CAPTURA DE FRAMES
frames = []
duration = 4.0   # Segundos de simulación
fps = 30         # Frames por segundo para el video

while data.time < duration:
    # Avanza la simulación física (calcula fuerzas, aceleraciones, integra en el tiempo)
    mujoco.mj_step(model, data)

    # Cada cierto tiempo, renderizamos un frame para el video
    if len(frames) < data.time * fps:
        renderer.update_scene(data)
        pixels = renderer.render()
        frames.append(pixels)

# 6. MOSTRAR EL VIDEO EN EL CUADERNO
print("Simulación completada. Renderizando video...")
media.show_video(frames, fps=fps)

Simulación completada. Renderizando video...


This browser does not support the video tag.
